In [1]:
import math
import os
import sys
sys.path.append(os.path.abspath('.')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
import gc
import catboost as cb
import lightgbm as lgb
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

from files_processor import LogFilesProcessor, WaferFilesProcessor, LogAndSpatialProcessor
from predictions import PrePredictionProcessor, SingleOutputModelPredictor, MultiOutputModelPredictor
from feature_selection import PCA_analysis, RFE_analysis
from asm_utils import Basics

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from autoencoder import Autoencoder, TrainAutoencoder

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))

import asm_data_wrangling as asm
from key_params import NUM_WAFERS, step_col_name, COMMON_ID_COLS, COMMON_ID_COLS_MOD, parquet_folder_name#, dict_of_spatial_files, dict_of_log_files 

log_processor   = LogFilesProcessor(COMMON_ID_COLS_MOD, COMMON_ID_COLS, parquet_folder_name)
multi_predictor = MultiOutputModelPredictor(device)
single_predictor= SingleOutputModelPredictor(device)


Using device: cpu


In [2]:
"""Constants"""

main_folder   = "../ASM_data"
dict_of_spatial_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                         'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}
marathon_run_col = "marathon_run"
wafer_col        = "wafer"
run_col          = "#run"
step_id_col      = "step_id"
process_time_col = "process time"
radius_col       = "Radius (mm)"
site_id_col      = "Site #"
spatial_property_col = "Spatial property (nm)"


In [3]:
"""code to split data into 4 wafers"""

# master_spatial_df, spatial_df_dict, y_df_dict, radius_wide_dict = asm.load_spatial_csv_and_create_targets(dict_of_spatial_files, main_folder, save=False)
# unique_marathon_runs_list = list(master_spatial_df["marathon_run"].unique())
# master_log_df = asm.load_and_process_and_combine_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
# # master_log_df = master_log_df.fill_null(pl.lit(0))
# master_log_df = remove_constant_valued_cols(master_log_df)
# master_log_df = master_log_df.fill_null(pl.lit(0))

# # asm.dont_split_log_df_by_wafer_and_save_to_parquet(master_log_df, main_folder, overwrite = True)
# # asm.split_log_df_by_wafer_and_save_to_parquet(master_log_df, NUM_WAFERS, main_folder, log_processor, overwrite = True)

# # log_df_with_wafer_col = asm.infer_wafer_from_rc_values(master_log_df, rc_prefix="rc", wafer_col="wafer", overwrite = False)
# # log_df_with_wafer_col = asm.fast_infer_wafer(master_log_df, NUM_WAFERS)


'code to split data into 4 wafers'

In [4]:
"""Code (new) to have a unified code/model (NOT split per wafer)"""

# spatial files
master_spatial_df, unique_marathon_runs_list = LogAndSpatialProcessor.load_spatial_csv_files_to_1_df(dict_of_spatial_files)
wide_radius_df, y_df = LogAndSpatialProcessor.create_target_df_from_spatial_df(master_spatial_df, main_folder, save=False)
# del master_spatial_df

# log files
master_log_df = log_processor.load_and_process_and_combine_log_csv_files(dict_of_log_files, unique_marathon_runs_list, step_col_name, main_folder, remove_runs_not_in_spatial_df=True, save_log_df_to_parquet = False)
master_log_df = Basics.remove_constant_valued_cols(master_log_df)
master_log_df = master_log_df.fill_null(pl.lit(0))
master_log_df_exploded = LogAndSpatialProcessor.explode_log_df_rows_by_wafer(master_log_df, NUM_WAFERS)
master_log_df_exploded = Basics.remove_constant_valued_cols(master_log_df_exploded)
del master_log_df


In [5]:
# =========== reshape master_log_df_exploded
# consider only 1 step
step_number    = 4
only_1_step_df = master_log_df_exploded.filter(pl.col(step_id_col) == step_number)

# subsample df
N_downsampling    = 5
subsampled_log_df = LogAndSpatialProcessor.downsample_df_rows(only_1_step_df, N_downsampling)

# latest rows per run:
num_last_rows_per_run       = 20
latest_rows_per_run_df      = (master_log_df_exploded
                               .sort(process_time_col)
                               .group_by([marathon_run_col, wafer_col])
                               .tail(num_last_rows_per_run))
latest_rows_per_step_run_df = (master_log_df_exploded
                               .sort(process_time_col)
                               .group_by([step_id_col, marathon_run_col, wafer_col])
                               .tail(num_last_rows_per_run))
reduced_log_df = latest_rows_per_run_df
# ===========

log_df_with_no_constant_cols  = Basics.remove_constant_valued_cols(reduced_log_df)
# why do we need the below line (add radius cols to X)? seems useless
# joined_log_df_with_spatial_df = LogAndSpatialProcessor.join_radius_df_to_exploded_log_df(log_df_with_no_constant_cols, wide_radius_df)
joined_log_df_with_spatial_df = log_df_with_no_constant_cols
y_df_expanded                 = LogAndSpatialProcessor.expand_y_df_to_match_size_of_log_df(joined_log_df_with_spatial_df, y_df)
joined_log_spatial_df_no_str  = joined_log_df_with_spatial_df.select(pl.exclude(pl.Utf8)) # remove str cols

#~~~~~~~~~~~~
# split then scale X (avoids data leakage)
preprocessor = PrePredictionProcessor()
X = joined_log_spatial_df_no_str.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col], errors='ignore')
y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
X_train, X_val, y_train, y_val  = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled, X_val_scaled, _ = preprocessor.scale_X_after_split(X_train, X_val)
#~~~~~~~~~~~~

try:
    del only_1_step_df, \
        master_log_df_exploded, subsampled_log_df, wide_radius_df, #log_df_with_no_constant_cols # y_df
        # joined_log_df_with_spatial_df,
except NameError:
    pass

In [ ]:
"""Keep y multi output"""

# Define cat_features before use
cat_features = [marathon_run_col, wafer_col]

X = joined_log_df_with_spatial_df.to_pandas()
X = X.drop(columns=process_time_col)
y = y_df_expanded.drop([marathon_run_col, wafer_col]).to_pandas()

# Split into numeric and categorical
X_cat = X[cat_features]
X_num = X.drop(columns=cat_features)

# Split indices once based on up-to-date data
train_idx, val_idx = train_test_split(X.index, test_size=0.2, random_state=42)

# Slice everything using consistent indices
X_num_train = X_num.loc[train_idx]
X_num_val   = X_num.loc[val_idx]
X_cat_train = X_cat.loc[train_idx]
X_cat_val   = X_cat.loc[val_idx]
y_train     = y.loc[train_idx]
y_val       = y.loc[val_idx]

# Scale only numeric
X_num_train_scaled, X_num_val_scaled, _ = preprocessor.scale_X_after_split(X_num_train, X_num_val)

# Combine scaled numeric with unscaled categoricals
X_train_scaled = pd.concat([X_num_train_scaled.reset_index(drop=True), X_cat_train.reset_index(drop=True)], axis=1)
X_val_scaled   = pd.concat([X_num_val_scaled.reset_index(drop=True), X_cat_val.reset_index(drop=True)], axis=1)

rmse_cat, y_pred_cat, _ = multi_predictor.predict_catboost(X_train_scaled, y_train.values, X_val_scaled,
                                                           y_val.values, cat_features=cat_features)
print(f"CatBoost RMSE: {rmse_cat:.5f} nm")


CatBoost RMSE (Selected Features): 0.68827 nm


In [ ]:
"""EXPLODE LOG DF and predict single-output y"""

# Ensure consistent dtypes
log_df   = log_df_with_no_constant_cols.with_columns([pl.col(wafer_col).cast(pl.Int32),
                                                      pl.col(marathon_run_col).cast(pl.Utf8)])
y_df     = master_spatial_df.with_columns([pl.col(wafer_col).cast(pl.Int32),
                                           pl.col(marathon_run_col).cast(pl.Utf8)])
y_subset = y_df.select([marathon_run_col, wafer_col, site_id_col, radius_col, spatial_property_col])

# Left join y_subset to exploded_df by matching sites per wafer
exploded_df = log_df.join(y_subset, on=[marathon_run_col, wafer_col], how="left")
y_expanded  = exploded_df.select([marathon_run_col, wafer_col, site_id_col, spatial_property_col]).rename({"Spatial property (nm)": "target"})

# Drop irrelevant cols from X and cast categoricals to str (for CatBoost)
X_full = exploded_df.drop([process_time_col, "#run", spatial_property_col]).with_columns([
    pl.col(wafer_col).cast(pl.Utf8),
    pl.col(site_id_col).cast(pl.Utf8)]).to_pandas()
y_full = y_expanded.select("target").to_pandas()

# Train/val split, then scale
X_train, X_val, y_train, y_val  = train_test_split(X_full, y_full, test_size=0.2, random_state=42)
X_train_scaled, X_val_scaled, _ = preprocessor.scale_X_after_split(X_train, X_val)

single_predictor = SingleOutputModelPredictor(device)

# === normal catboost pred ====
cat_features = [wafer_col, site_id_col, marathon_run_col]
rmse, y_pred, importances = single_predictor.predict_catboost_single_model(
    X_train_scaled, y_train, X_val_scaled, y_val, cat_features=cat_features)
print(f"CatBoost RMSE (all features): {rmse:.5f} nm")

numeric_feature_names = X_train.select_dtypes(include=np.number).columns
# X_train_num = X_train_scaled[numeric_feature_names]
# X_val_num   = X_val_scaled[numeric_feature_names]

X_train_num       = X_train_scaled.select_dtypes(include=np.number)
X_val_num         = X_val_scaled.select_dtypes(include=np.number)


CatBoost RMSE (all features): 1.14178 nm


In [ ]:
"""PCA + kmeans"""

# ===== PCA =====
pca_object = PCA_analysis()
pca_model  = pca_object.fit_pca(X_train_scaled[numeric_feature_names], var_threshold=0.99)

# Explained variance info
components, n_components = pca_object.explain_pca_variance(pca_model, show_plot=False)

# Transformed data already reduced to n_components
X_train_pca_reduced = pca_model.transform(X_train_scaled[numeric_feature_names])
X_val_pca_reduced   = pca_model.transform(X_val_scaled[numeric_feature_names])

rmse_pca, y_pred_pca, _ = single_predictor.predict_catboost_single_model(
    X_train_pca_reduced, y_train, X_val_pca_reduced, y_val, cat_features=None)
print(f"CatBoost RMSE (PCA reduced): {rmse_pca:.3f} nm")

# ===== SelectKBest =====
k_features = len(numeric_feature_names)
selector   = SelectKBest(score_func=f_regression, k = k_features)

X_train_selected = selector.fit_transform(X_train_num, y_train.values.ravel())
X_val_kbest      = selector.transform(X_val_scaled[numeric_feature_names])

selected_cols = X_train_num.columns[selector.get_support()]
# print(f"Selected features (SelectKBest): {selected_cols.tolist()}")

rmse_kbest, y_pred_kbest, _ = single_predictor.predict_catboost_single_model(
    X_train_selected, y_train, X_val_kbest, y_val, cat_features=None)
print(f"CatBoost RMSE (SelectKBest): {rmse_kbest:.3f} nm")

# ===== Kbest + PCA =======
# # Step 1: Select K best features
# k_features = len(numeric_feature_names)
# selector   = SelectKBest(score_func=f_regression, k = k_features)

# X_train_kbest = selector.fit_transform(X_train_scaled[numeric_feature_names], y_train.values.ravel())
# X_val_kbest   = selector.transform(X_val_scaled[numeric_feature_names])

# # Step 2: PCA on K-best features using your PCA_analysis class
# pca_object = PCA_analysis()
# pca_model  = pca_object.fit_pca(pd.DataFrame(X_train_kbest), var_threshold=0.99)

# X_train_kbest_pca = pca_model.transform(X_train_kbest)
# X_val_kbest_pca   = pca_model.transform(X_val_kbest)

# _, N_pca_components = pca_object.explain_pca_variance(pca_model, show_plot=False)

# # Step 3: CatBoost prediction
# rmse_kbest_pca, y_pred_kpca, _ = single_predictor.predict_catboost_single_model(
#     X_train_kbest_pca, y_train, X_val_kbest_pca, y_val, cat_features=None)
# print(f"CatBoost RMSE (KBest + PCA): {rmse_kbest_pca:.3f} nm")


In [ ]:
"""New methods"""

from catboost import CatBoostRegressor
# from sklearn.feature_selection import SelectFromModel

# === Feature selection using CatBoost (includes categoricals) ===
catboost_model = CatBoostRegressor(verbose=0, random_state=42)
catboost_model.fit(X_train_scaled, y_train, cat_features = cat_features)

# Get feature importances
importances      = catboost_model.get_feature_importance()
feature_names    = X_train_scaled.columns
thres_percent    = 25 #17 # keep top (100-thres_percent)% features
selected_mask    = importances >= np.percentile(importances, thres_percent)

selected_columns = feature_names[selected_mask]
X_train_reduced  = X_train_scaled[selected_columns]
X_val_reduced    = X_val_scaled[selected_columns]

# Recompute categorical feature indices post-selection
cat_features_reduced = [i for i, col in enumerate(selected_columns) if col in cat_features]

# Predict with reduced feature set
rmse_sfm, _, _ = single_predictor.predict_catboost_single_model(
    X_train_reduced, y_train, X_val_reduced, y_val, cat_features=cat_features_reduced)
print(f"CatBoost RMSE (Selected Features): {rmse_sfm:.5f} nm")


sys.exit()

# Get sorted features by importance
support_mask    = selector.get_support()
importances     = selector.estimator_.feature_importances_
selected_cols   = X_train_num.columns[support_mask]
sorted_indices  = importances[support_mask].argsort()[::-1]
sorted_features = selected_cols[sorted_indices]
print(sorted_features)

sys.exit()

# ====== recursive feature elimination (RFE) with catboost ==========
X_train_num         = X_train_scaled.select_dtypes(include=np.number)
RFE_object          = RFE_analysis(device)
rmse_rfe, rfe_model = RFE_object.apply_recursive_feature_elimination(X_train_scaled, X_val_scaled, y_train, y_val, single_predictor, fraction_cols_to_keep = 0.99)
sorted_features     = RFE_object.get_sorted_features_by_importance(rfe_model, X_train_num)


# cached_rfe = {"selected_features": sorted_features.tolist(),
#               "importances": importances.tolist(),
#               "sorted_features": sorted_features.tolist(),
#               "estimator": rfe_model}
# import joblib
# joblib.dump(cached_rfe, "rfe_pass1_results.pkl")
# cached_rfe = joblib.load("rfe_pass1_results.pkl")


# pass #2
X_train_reduced     = X_train_scaled[sorted_features]
X_val_reduced       = X_val_scaled[sorted_features]
X_train_num_reduced = X_train_reduced.select_dtypes(include=np.number)
rmse_rfe, rfe_model = RFE_object.apply_recursive_feature_elimination(X_train_reduced, X_val_reduced, y_train,
                                                                     y_val, single_predictor, fraction_cols_to_keep=0.5)
sorted_features2    = RFE_object.get_sorted_features_by_importance(rfe_model, X_train_num_reduced)


# # ======== Drop Low-Importance Features (CatBoost)====
# model = CatBoostRegressor(verbose=0, random_state=42)
# model.fit(X_train_scaled, y_train, cat_features=cat_features)

# importances = model.get_feature_importance()
# keep_mask   = importances > np.percentile(importances, 25)

# X_train_imp = X_train_scaled.iloc[:, keep_mask]
# X_val_imp   = X_val_scaled.iloc[:, keep_mask]

# rmse_imp, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_imp, y_train, X_val_imp, y_val, cat_features=None)
# print(f"CatBoost RMSE (Important features): {rmse_imp:.3f} nm")

# # ====== UMAP ========
# import umap
# X_train_num = X_train_scaled[numeric_feature_names]
# X_val_num   = X_val_scaled[numeric_feature_names]

# umap_model = umap.UMAP(n_components=10, random_state=42)
# X_train_umap = umap_model.fit_transform(X_train_num)
# X_val_umap   = umap_model.transform(X_val_num)

# rmse_umap, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_umap, y_train, X_val_umap, y_val, cat_features=None)
# print(f"CatBoost RMSE (UMAP): {rmse_umap:.3f} nm")

# # ======= Polynomial Features + KBest =======
# from sklearn.preprocessing import PolynomialFeatures

# X_train_num = X_train_scaled[numeric_feature_names]
# X_val_num   = X_val_scaled[numeric_feature_names]

# poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
# X_train_poly = poly.fit_transform(X_train_num)
# X_val_poly   = poly.transform(X_val_num)

# selector = SelectKBest(score_func=f_regression, k=50)
# X_train_poly_sel = selector.fit_transform(X_train_poly, y_train.values.ravel())
# X_val_poly_sel   = selector.transform(X_val_poly)

# rmse_poly, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_poly_sel, y_train, X_val_poly_sel, y_val, cat_features=None)
# print(f"CatBoost RMSE (Poly + SelectKBest): {rmse_poly:.3f} nm")


In [ ]:
print(sorted_features)

plt.figure(figsize=(16, 14))
for col in sorted_features:
    if (col not in log_df_with_no_constant_cols.columns) or (col.startswith("rc")):
        continue
    s = log_df_with_no_constant_cols[col]
    plt.plot(s, label=col, linewidth=2)
plt.legend(fontsize=10)
plt.show()

plt.figure(figsize=(16, 14))
for col in sorted_features:
    if (col not in log_df_with_no_constant_cols.columns):
        continue
    if (col.startswith("rc")):
        s = log_df_with_no_constant_cols[col]
        plt.plot(s, label=col, linewidth=2)
plt.legend(fontsize=10)
plt.show()


In [ ]:
"""Autoencoder"""

this_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device from module: {this_device}")

layer1_dim=          128         # layer 1 nodes
layer2_dim=          64          # layer 2 nodes
latent_dim=          32          # num of features in latent layer, get this from data dimensionality
dropout_prob=        0.05
ae_training_epochs=  100        # training epochs
ae_batch_size=       126        # number of samples per batch
ae_optimizer_lr=     0.0011     # learning rate for the optimizer
weight_decay=        0.00001    # for L2 regularization
training_patience=   80         # how many epochs with no improvement to stop training
scheduler_patience=  80
scheduler_mode=      'min'      # min (max) reduces elarning rate when validation loss stops improving (starts increasing)
scheduler_factor=    0.8        # multiplies lr by this factor when validation loss plateaus

reduced_log_df_numeric = reduced_log_df.select(pl.col(pl.NUMERIC_DTYPES))

X_scaled_tensor = torch.tensor(X_scaled.values, dtype=torch.float32)
X_train_torch   = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_val_torch     = torch.tensor(X_val.to_numpy(),   dtype=torch.float32)

train_dataset = TensorDataset(X_train_torch, torch.zeros(len(X_train)))  # dummy labels
val_dataset   = TensorDataset(X_val_torch, torch.zeros(len(X_val)))

# Create DataLoader objects for train and validation datasets
input_size   = X_train_torch.shape[1]
train_loader = DataLoader(train_dataset, batch_size=ae_batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=ae_batch_size, shuffle=False)

autoencoder  = Autoencoder(input_size, layer1_dim, layer2_dim, latent_dim, dropout_prob)
optimizer    = torch.optim.AdamW(autoencoder.parameters(), lr=ae_optimizer_lr, weight_decay=weight_decay)
scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, scheduler_mode, patience=scheduler_patience, factor=scheduler_factor)

if not os.path.exists('autoencoder.pth'):
    trainer   = TrainAutoencoder()
    best_loss = trainer.train_autoencoder(this_device, autoencoder, ae_training_epochs, train_loader, optimizer, scheduler,
                                          validation_loader=val_loader, patience = training_patience)
    print(f"best loss: {best_loss:.3f}")

    # save trained AE
    torch.save(autoencoder.state_dict(), 'autoencoder.pth')


In [ ]:
"""Latents"""

from sklearn.decomposition import PCA

# load trained AE

autoencoder = Autoencoder(input_size, layer1_dim, layer2_dim, latent_dim, dropout_prob)
autoencoder.load_state_dict(torch.load('autoencoder.pth'))
autoencoder.to(this_device)
autoencoder.eval()

# Get latent space for X_train
with torch.no_grad():
    X_latent: np.ndarray       = autoencoder.encoder(X_scaled_tensor.to(this_device)).cpu().numpy()
    X_train_latent: np.ndarray = autoencoder.encoder(X_train_torch.to(this_device)).cpu().numpy()
    X_val_latent: np.ndarray   = autoencoder.encoder(X_val_torch.to(this_device)).cpu().numpy()
    
# 2D
plt.scatter(X_latent[:, 0], X_latent[:, 1], s=3, alpha=0.5)
plt.title("2D Latent Space")
plt.xlabel("Latent dim 1")
plt.ylabel("Latent dim 2")
plt.show()

# 3D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_latent)

plt.scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.5)
plt.title("Latent Space (PCA to 2D)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.show()


In [ ]:
"""Decoder + PCA"""

# X_train_latent[:, 8] = 0
X_train_recon = autoencoder.decoder(torch.from_numpy(X_train_latent).to(this_device)).detach().cpu().numpy()
X_val_recon   = autoencoder.decoder(torch.from_numpy(X_val_latent).to(this_device)).detach().cpu().numpy()
X_recon       = autoencoder.decoder(torch.from_numpy(X_latent).to(this_device)).detach().cpu().numpy()
# ===

latent_df = pd.DataFrame(X_scaled, columns=[f"z{i}" for i in range(X_scaled.shape[1])])
latent_dim = X_scaled.shape[1]
# latent_df = pd.DataFrame(X_latent, columns=[f"z{i}" for i in range(X_latent.shape[1])])
target_df = pd.DataFrame(y_scaled, columns=[f"y{i}" for i in range(y_scaled.shape[1])])

# corr_matrix = latent_df.corr()
# sns.heatmap(corr_matrix.abs(), cmap='viridis')

corr_matrix = latent_df.corrwith(target_df, axis=0)# This won't work directly because corrwith compares series by index, not cross-columns.

# Instead, compute pairwise correlations manually:
corr_matrix = pd.DataFrame(
    np.corrcoef(latent_df.values.T, target_df.values.T)[:latent_dim, latent_dim:],
    index=latent_df.columns,
    columns=target_df.columns)

sns.heatmap(corr_matrix.abs(), cmap='viridis')
plt.xlabel('Targets')
plt.ylabel('Latent Features')
plt.show()


In [ ]:
"""Correlation of X with y"""

if not isinstance(X_scaled, pd.DataFrame):
    X_df = pd.DataFrame(X_scaled, columns=[f"x{i}" for i in range(X_scaled.shape[1])])
else:
    X_df = X_scaled.copy()

target_df = pd.DataFrame(y_scaled, columns=[f"y{i}" for i in range(y_scaled.shape[1])])

# Compute correlation matrix
corr_matrix = pd.DataFrame(
    np.corrcoef(X_df.values.astype(np.float64).T, target_df.values.astype(np.float64).T)[:X_df.shape[1], X_df.shape[1]:],
    index=X_df.columns,
    columns=target_df.columns)

# Get top input features by max absolute correlation across targets
top_N_features  = 50
avg_corr_per_feature = corr_matrix.abs().mean(axis=1)
top_features    = avg_corr_per_feature.nlargest(top_N_features).index
corr_matrix_top = corr_matrix.loc[top_features]

# Plot heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(corr_matrix_top.abs(), cmap='coolwarm')
plt.xlabel('Targets')
plt.ylabel(f'Top {top_N_features} Input Features')
plt.yticks(ticks=np.arange(corr_matrix_top.shape[0]), labels=corr_matrix_top.index, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
"""Kmeans clustering. NOTE make sure to apply it to the entire X dataset, not just runs which have a wafer"""

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42)
cluster_labels = kmeans.fit_predict(X_latent)

# # ===================
# plot 1
# X_pca = PCA(n_components=2).fit_transform(X_latent)
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='Set1', s=5)
# plt.title("Clustering in Latent Space")
# plt.show()

# # ===================
# plot 1.2
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=10, max_iter=1500)
X_tsne = tsne.fit_transform(X_latent)

plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='Set1', s=5)
plt.title("Clustering in Latent Space (t-SNE)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

# # ===================
# # plot 1.2
# import umap.umap_ as umap  # if you installed umap-learn

# # Fit UMAP to reduce latent space to 2D
# reducer = umap.UMAP(n_components=2, random_state=42)
# X_umap = reducer.fit_transform(X_latent)

# # KMeans clustering (same as before)
# kmeans = KMeans(n_clusters=12, random_state=42)
# cluster_labels = kmeans.fit_predict(X_latent)

# # Plot UMAP projection with cluster colors
# plt.scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels, cmap='Set1', s=5)
# plt.title("Clustering in Latent Space (UMAP)")
# plt.xlabel("UMAP 1")
# plt.ylabel("UMAP 2")
# plt.show()


In [ ]:
"""[PCA] High-variance features are NOT always predictive, use PCA components as a safe bet.
PCA components = linear combinations of features, capture global structure, NOT local feature importance
NOTE that PCA does not look at y, only X"""

pca_obj   = PCA_analysis()
pca_model = pca_obj.fit_pca(X_scaled)
X_pca     = pca_model.transform(X_scaled)

top_N_pca_components, N_pca_components = pca_obj.explain_pca_variance(pca_model, var_threshold=0.99)
X_pca_reduced = X_pca[:, :N_pca_components]

pca_obj.print_top_features_per_component(X_scaled, top_N_pca_components, top_k_features=2)
sorted_features = pca_obj.summarize_feature_importance(X_scaled, top_N_pca_components, top_k_features=5)


In [ ]:
"""SimCLR"""

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Dataset with augmentations (simple: random noise)
class SimCLRDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def augment(self, x):
        # Dropout (feature masking)
        mask_prob = 0.05
        mask = (torch.rand_like(x) > mask_prob).float()
        x_masked = x * mask

        # Jitter (feature scaling)
        scale = 0.9 + 0.2 * torch.rand_like(x)
        x_scaled = x_masked * scale

        # Noise (gaussian noise)
        noise = 0.05 * torch.randn_like(x_masked)
        return x_scaled + noise

    def __getitem__(self, idx):
        x = self.X[idx]
        return self.augment(x), self.augment(x)

# Simple MLP encoder for tabular data
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.net(x)

# Projection head as in SimCLR paper
class ProjectionHead(nn.Module):
    def __init__(self, latent_dim=128, proj_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, proj_dim)
        )
    def forward(self, x):
        return self.net(x)

# NT-Xent loss for SimCLR
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)

    # Mask to ignore similarity with self
    mask = (~torch.eye(2*batch_size, 2*batch_size, dtype=bool)).to(z1.device)

    positives = torch.cat([torch.diag(similarity_matrix, batch_size),
                           torch.diag(similarity_matrix, -batch_size)], dim=0)

    negatives = similarity_matrix[mask].view(2*batch_size, -1)

    logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
    labels = torch.zeros(2*batch_size, dtype=torch.long).to(z1.device)

    logits = logits / temperature
    loss = F.cross_entropy(logits, labels)
    return loss

# Usage example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_scaled_tensor = torch.tensor(X_scaled.values, dtype=torch.float32)
y_scaled_tensor = torch.tensor(y_scaled, dtype=torch.float32)

dataset = SimCLRDataset(X_scaled_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

input_dim = X_scaled_tensor.shape[1]
encoder = Encoder(input_dim).to(device)
proj_head = ProjectionHead().to(device)
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(proj_head.parameters()), lr=0.0015)

for epoch in range(10):
    total_loss = 0
    for x1, x2 in dataloader:
        x1, x2 = x1.to(device), x2.to(device)
        h1 = encoder(x1)
        h2 = encoder(x2)
        z1 = proj_head(h1)
        z2 = proj_head(h2)
        loss = nt_xent_loss(z1, z2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")


In [ ]:
"""t-SNE"""

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

encoder.eval()
with torch.no_grad():
    embeddings = encoder(torch.tensor(X_scaled.values, dtype=torch.float32).to(device)).cpu().numpy()

tsne = TSNE(n_components=2, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y_scaled_tensor.argmax(dim=1).cpu().numpy(), cmap='tab10', s=5)
plt.title('TSNE of SimCLR Embeddings')
plt.show()


In [ ]:

# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_train_latent, y_train, X_val_latent, y_val)
# print(rmse_cat)#, y_pred_cat)

# Get error on RECONSTRUCTED X (using autoencoder)
# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_train_recon, y_train, X_val_recon, y_val)
# print(rmse_cat)#, y_pred_cat)

rmse, y_pred, importances = single_predictor.predict_catboost_single_model(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, cat_features=cat_features)
print(rmse)#, y_pred_cat)

# X_pca_train, X_pca_val, y_train, y_val = train_test_split(X_pca_reduced, y_scaled, test_size=0.2, random_state=42)
# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_pca_train, y_train, X_pca_val, y_val)
# print(rmse_cat)#, y_pred_cat)

# Conclusion: compressing to latent, zeroing the least correlated latent col, then decompressing and predicting yields WORSE score

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_models_in_1_dataset(y_df_expanded, joined_log_spatial_df, main_folder, device):
    """Edited to do everything in 1 dataset"""
    marathon_run_col = "marathon_run"
    wafer_col        = "wafer"
    run_col          = "#run"

    # convert wafer col to str, maybe it helps:
    # joined_log_spatial_df = joined_log_spatial_df.with_columns(pl.col(wafer_col).cast(str))

    preprocessor = PrePredictionProcessor()

    X_train: pd.DataFrame
    y_train: np.ndarray
    X_val:   pd.DataFrame
    y_val:   np.ndarray

    want_to_scale_per_wafer = False
    if want_to_scale_per_wafer:
        X = joined_log_spatial_df.to_pandas().drop(columns=[marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col).to_pandas()
        X_train, y_train, X_val, y_val, wafer_x_scalers, wafer_y_scalers = preprocessor.scale_per_wafer_and_split_data(X, y, wafer_col="wafer", test_size=0.2)
    else:
        X = joined_log_spatial_df.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
        # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
        X_scaled, y_scaled, _ = preprocessor.scale_X_after_split(X, y)
        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)

    numeric_cols  = X_train.select_dtypes(include=[np.number]).columns
    zero_var_cols = X_train[numeric_cols].columns[X_train[numeric_cols].var() == 0].tolist()
    X_train_clean = X_train.drop(columns = zero_var_cols)
    X_val_clean   = X_val.drop(columns = zero_var_cols)

    # =-=-=-=-=-= bit about feature importance
    importances = multi_predictor._make_predictions_in_1_function(X_train_clean, y_train, X_val_clean, y_val, device)

    top_features_fraction = 0.05
    X_train_clean_light, X_val_clean_light = LogAndSpatialProcessor.keep_top_features_by_importance(X_train_clean, X_val_clean, importances, top_features_fraction)
    
    # Cross Val score
    model  = RandomForestRegressor()
    scores = cross_val_score(model, X_train_clean_light, y_train, scoring=rmse_scorer, cv=5)
    print(f"CV RMSE mean: {-scores.mean():.3f}")
    
    importances = multi_predictor._make_predictions_in_1_function(X_train_clean_light, y_train, X_val_clean_light, y_val, device)

    # =-=-=-=-=-=-=-=-=-=-=-=

    # ========= bit about correlation
    # correlations = pd.DataFrame({
    #     f"target_{i}": X_train_clean.corrwith(pd.Series(y_train[:, i], index=X_train_clean.index)).abs()
    #     for i in range(y_train.shape[1])})

    # # Average correlations across all targets
    # mean_correlations = correlations.mean(axis=1)

    # # Model importances (make sure index aligns with X_train_clean.columns)
    # importances_mean = pd.Series(mean_importance, index=X_train_clean.columns)

    # # Plot correlation vs importance
    # plt.figure(figsize=(8,6))
    # plt.scatter(mean_correlations, importances_mean)
    # plt.xlabel("Mean Abs(Correlation) with Targets")
    # plt.ylabel("Mean Model Feature Importance")
    # plt.title("Feature Importance vs. Correlation")
    # plt.grid(True)
    # plt.show()
    # ============

    return importances

importances = train_models_in_1_dataset(y_df_expanded, joined_log_spatial_df_no_str, main_folder, device)


In [ ]:

plt.figure(figsize=(17, 8))
# y_values = joined_log_spatial_df_no_str['rc3 signal_1_step4']
y_values = joined_log_spatial_df_no_str['common signal_88_step4']
x_values = range(len(y_values))
plt.scatter(x_values, y_values)
plt.show()


In [ ]:
which_row = 1
y_real_values = (y_df_expanded.row(which_row))[2:]

y_pred_cat_original_scale = y_scaler.inverse_transform(y_pred_cat)
y_val_original_scale = y_scaler.inverse_transform(y_val)
y_pred_values = y_val_original_scale[which_row]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(range(len(y_real_values)), y_real_values, label='real')
ax.scatter(range(len(y_pred_values)), y_pred_values, label='predicted')
ax.set_xlabel("Site ID")
ax.set_ylabel("Spatial property (nm)")
ax.set_title("Spatial property, real vs predicted")
ax.legend()
plt.show()


In [ ]:
def join_features_targets(big_log_df, target_df):
    # target columns except marathon_run
    target_cols = [str(i) for i in range(1, 110)]
    
    # merge on marathon_run
    full_df = big_log_df.merge(target_df, on='marathon_run', how='left')
    
    # features: drop target columns + marathon_run if not needed as feature
    X = full_df.drop(columns=target_cols + ['marathon_run'])
    
    # targets
    y = full_df[target_cols]
    
    return X, y

# to fix, better to have the object passed onto the function
def train_one_model_for_all_wafers(y_df_dict, radius_wide_dict, big_log_df, marathon_run_col, target_cols, device):
    predictor    = MultiOutputModelPredictor(device)
    preprocessor = PrePredictionProcessor()
    wafer_col    = "wafer"

    # Combine all y dfs into one with wafer column
    y_dfs = []
    for k, df in y_df_dict.items():
        y_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_y_df = pl.concat(y_dfs, how="vertical")

    # Combine all radius dfs into one with wafer column
    radius_dfs = []
    for k, df in radius_wide_dict.items():
        radius_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_radius_df = pl.concat(radius_dfs, how="vertical")

    # Flatten last rows of big_log_df by marathon_run
    processed_log_df = asm._flatten_last_n_rows(big_log_df, marathon_run_col, num_of_last_rows=1)

    # Join features with radius info on marathon_run and wafer
    features_df = processed_log_df.join(
        combined_radius_df,
        on  = [marathon_run_col, wafer_col],
        how = "left")

    # Join features with targets on marathon_run and wafer
    full_df = features_df.join(
        combined_y_df,
        on  = [marathon_run_col, wafer_col],
        how = "inner")

    # Prepare X and y
    y = full_df.select(target_cols).to_pandas()
    X = full_df.drop(target_cols + [marathon_run_col, wafer_col]).to_pandas()

    X = preprocessor.drop_certain_cols_from_df(X, [marathon_run_col])

    # Scale and split
    # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
    X_scaled, y_scaled, _ = preprocessor.scale_X_after_split(X, y)
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)
    X_train = X_train.fillna(0)
    X_val   = X_val.fillna(0)

    # Drop zero variance cols
    zero_var_cols = X_train.columns[X_train.var() == 0].tolist()
    X_train = X_train.drop(columns=zero_var_cols)
    X_val   = X_val.drop(columns=zero_var_cols)

    # Train all models and print results
    rmse_linreg, _ = predictor.predict_linear_reg(X_train, y_train, X_val, y_val)
    rmse_ridge, _  = predictor.predict_linear_reg_ridge(X_train, y_train, X_val, y_val)
    rmse_lgb, _    = predictor.predict_lightgbm(X_train, y_train, X_val, y_val)
    # rmse_cat, _ = predictor.predict_catboost(X_train, y_train, X_val, y_val)
    rmse_rf, _     = predictor.predict_randomforest(X_train, y_train, X_val, y_val)

    print(f"LinReg RMSE: {rmse_linreg:.3f}")
    print(f"Ridge RMSE: {rmse_ridge:.3f}")
    print(f"LGBM RMSE: {rmse_lgb:.3f}")
    # print(f"Catboost RMSE: {rmse_cat:.3f}")
    print(f"RF RMSE: {rmse_rf:.3f}")

    return {"linreg_rmse": rmse_linreg,
            "ridge_rmse":  rmse_ridge,
            "lgbm_rmse":   rmse_lgb,
            # "catboost_rmse": rmse_cat,
            "rf_rmse":     rmse_rf,}

target_cols = [str(i) for i in range(1, 110)]  # '1' to '109'

results = train_one_model_for_all_wafers(
    y_df_dict        = y_df_dict,
    radius_wide_dict = radius_wide_dict,
    big_log_df       = big_log_df,
    marathon_run_col = "marathon_run",
    target_cols      = target_cols,
    device           = device)

print(results)

# combined_log_df["wafer"]
# print(y_df_dict[0].columns)


In [ ]:

def find_square_periodicity_of_feature(signal):
    """Requires a square function"""
    durations   = np.diff(np.where(np.diff(signal) != 0)[0])
    periods     = durations[::2] + durations[1::2]  # Sum long+short phases
    estimated_p = int(np.round(np.median(periods)))
    return estimated_p


In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()